In [1]:
%pip install -q -U google-genai chromadb pypdf


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os

from dotenv import load_dotenv
load_dotenv(override=True)  # Load environment variables from .env file

api_key = os.getenv("GOOGLE_API_KEY")

print(len(api_key))

53


**Part A — Build the index:** PDF → page text → overlapping chunks → embeddings → Chroma.

**Part B — Answer a question:** question → query embedding → matching chunks → prompt → answer.

## PDF

In [8]:
from pathlib import Path 

In [10]:
file_path = Path("leave_no_context_behind.pdf")
CHUNKSIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5
DB_PATH = "./chroma_db"


if not file_path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")
else:
    print(f"File found: {file_path}")

File found: leave_no_context_behind.pdf


# PART A

## - READ each page - 

A page is an ordinary dictionary with its text, source filename, and page number. Keeping the page number lets us trace retrieved passages back to the PDF.

`pypdf` extracts existing text; it does not perform OCR on scanned images. Pages without extractable text are skipped.

In [18]:
from pypdf import PdfReader
pages = []

reader = PdfReader(file_path)

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    if text:
        pages.append({
            "text:": text,
            "page_number": page_number,
            "source": file_path.name
        })

## Split the text into overlapping chunks

Searching smaller passages is more useful than retrieving a whole PDF. Each chunk contains at most 1,000 **characters**, with 200 characters shared with the next chunk.

For example, the first chunk covers characters 0–999; the next starts at character 800. Overlap helps preserve information near a boundary.

This simple splitter uses character positions, so it can cut through words or sentences. It splits each page separately, making page references easy to follow. Characters are different from model tokens.

In [ ]:
text = (
    "Lorem ipsum dolor sit amet, consectetur adipiscing elit. "
    "Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. "
    "Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi "
    "ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit "
    "in voluptate velit esse cillum dolore eu fugiat nulla pariatur. "
)
strat = 0
end = min(len(text), 50)

chunk1 = text[strat:end]
chunk2 = text[end:end + 50]   
chunk3 = text[end + 50:end + 100]



In [ ]:
text = "I have this text that I want to embed and store in a vector database. make it chars"

In [27]:
print(f"Number of pages: {len(pages)}")

Number of pages: 14


In [28]:
def split_text(text, chunk_size=1000, overlap=200):
    pieces = []
    start = 0
    while start < len(text):
        end = min(start+ chunk_size, len(text))
        piece = text[start:end].strip()
        if piece:
            pieces.append(piece)
        if end ==len(text):
            break
        start += chunk_size - overlap
    return pieces

chunks = []
for page in pages:
    for text in split_text(page["text:"], chunk_size=CHUNKSIZE, overlap=CHUNK_OVERLAP):
        chunks.append({
            "text:": text,
            "page_number": page["page_number"],
            "source": page["source"]
        })




In [29]:
print(f"number of chunks: {len(chunks)}")


number of chunks: 60


In [26]:
# pages[0]["text:"][:20]

'Preprint. Under revi'

In [ ]:
# for page in reader.pages:
#     text = page.extract_text()
#     print(text)

#     break

## Turn text into Embeddings


In [51]:
chunks[0]

{'text:': 'Preprint. Under review.\nLeave No Context Behind:\nEfficient Infinite Context Transformers with Infini-attention\nTsendsuren Munkhdalai, Manaal Faruqui and Siddharth Gopal\nGoogle\ntsendsuren@google.com\nAbstract\nThis work introduces an efficient method to scale Transformer-based Large\nLanguage Models (LLMs) to infinitely long inputs with bounded memory\nand computation. A key component in our proposed approach is a new at-\ntention technique dubbed Infini-attention. The Infini-attention incorporates\na compressive memory into the vanilla attention mechanism and builds\nin both masked local attention and long-term linear attention mechanisms\nin a single Transformer block. We demonstrate the effectiveness of our\napproach on long-context language modeling benchmarks, 1M sequence\nlength passkey context block retrieval and 500K length book summarization\ntasks with 1B and 8B LLMs. Our approach introduces minimal bounded\nmemory parameters and enables fast streaming inferenc

In [52]:
embedding_func = embedding_functions.DefaultEmbeddingFunction()

chroma_client = chromadb.PersistentClient(path=DB_PATH)

collection = chroma_client.create_collection(name="pdf_collection",
 embedding_function=embedding_func,
 metadata={"hnsw:space": "cosine"})    


In [53]:

 collection.add(
    ids=[f"id-{i}" for i in range(len(chunks))],
    documents=[chunk["text:"] for chunk in chunks],
    metadatas=[{"page_number": chunk["page_number"], "source": chunk["source"]} for chunk in chunks]
    )


In [55]:
def retrieve_chunks(question, top_k=TOP_K):

    query_results = collection.query(
        query_texts=[question],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    matches = []
    for text, metadata, distance in zip(
        query_results["documents"][0],
        query_results["metadatas"][0],
        query_results["distances"][0]
    ):
        matches.append(({"text": text,
                        "page": metadata["page_number"],
                        "source": metadata["source"],
                        "distance": distance  }))
    return matches

In [60]:
def build_context(matches):
    context = ""
    for match in matches:
        context += f"Page {match['page']} (source: {match['source']}):\n{match['text']}\n\n"
    return context.strip()


def build_prompt(question, context):
    prompt = f"""You are a helpful assistant.
                Use the following context to answer the question.
                If the context does not contain the answer, say "I don't know".
                Context:
                \n
                ```{context}```
                Question: \n\n"{question}"
                Answer:"""
    return prompt.strip()


context = build_context(retrieve_chunks("What is the main topic of the document?"))
prompt = build_prompt("What is the main topic of the document?", context)

print(prompt)

You are a helpful assistant.
                Use the following context to answer the question.
                If the context does not contain the answer, say "I don't know".
                Context:
                

                ```Page 8 (source: leave_no_context_behind.pdf):
d on a book summarization task,
BookSum (Kry´sci ´nski et al., 2021) where the
goal is to generate a summary of an entire
book text.
We set the input length to 32K for fine-tuning and increase to 500K for evaluating. We use a
generation temperature of 0.5 and top p = 0.95 and set the number of decoding steps to 1024
to generate a summary of each book.
Table 4 compares our model against the encoder-decoder models that were built particularly
for the summarization task (Lewis et al., 2019; Xiao et al., 2021) and their retrieval-based
long-context extension (Bertsch et al., 2024). Our model outperforms the previous best
8

Page 14 (source: leave_no_context_behind.pdf):
Preprint. Under review.
B Passkey Retrieva

In [ ]:
from google import genai

client = genai.Client(
    api_key=GOOGLE_API_KEY
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="what is Deep learning?"
)

print(response.text)

In [68]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

def generate_answer(prompt):
    from google import genai
    client = genai.Client(
        api_key=GOOGLE_API_KEY
    )
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text

In [ ]:
def ask_pdf(question, top_k=TOP_K):
    matches = retrieve_chunks(question)
    context = build_context(matches)
    prompt = build_prompt(question, context)
    answer = generate_answer(prompt)
    return answer

print(ask_pdf("What is the main topic of the document?"))

In [ ]:
1. load_and_chunk_document(document_path)
2. retrieve_sections(question, embedding_model, collection)
3. augment_prompt(question, retrieved_chunks)
4. generate_answer(prompt, client)
5. regulations_qa_pipeline(question, document_path)


In [32]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

default_ef = DefaultEmbeddingFunction()
embeddings = default_ef(["faaahhhh", "faaahhhh2"])

print(len(embeddings)) # 2
# print(embeddings) # [[0.05035809800028801, 0.0626462921500206, -0.061827320605516434...]]

# collection.query(query_embeddings=embeddings)

2


In [ ]:
import chromadb
from chromadb.utils import embedding_functions



In [43]:
import chromadb
from chromadb.utils import embedding_functions

CHROMA_DATA_PATH = "chroma_data/"
EMBED_MODEL = "all-MiniLM-L6-v2"
COLLECTION_NAME = "demo_docs"

chroma_client = chromadb.PersistentClient(path=CHROMA_DATA_PATH)

In [44]:
embedding_func = embedding_functions.DefaultEmbeddingFunction()
import time 

collection = chroma_client.create_collection(name=COLLECTION_NAME+str(time.time()),
        embedding_function=embedding_func,
        metadata={"hnsw:space": "cosine"}
)

In [45]:
documents = [
    "The latest iPhone model comes with impressive features and a powerful camera.",
    "Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.",
    "Einstein's theory of relativity revolutionized our understanding of space and time.",
    "Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.",
    "The American Revolution had a profound impact on the birth of the United States as a nation.",
    "Regular exercise and a balanced diet are essential for maintaining good physical health.",
    "Leonardo da Vinci's Mona Lisa is considered one of the most iconic paintings in art history.",
    "Climate change poses a significant threat to the planet's ecosystems and biodiversity.",
    "Startup companies often face challenges in securing funding and scaling their operations.",
    "Beethoven's Symphony No. 9 is celebrated for its powerful choral finale, 'Ode to Joy.'",
]

genres = [
    "technology",
    "travel",
    "science",
    "food",
    "history",
    "fitness",
    "art",
    "climate change",
    "business",
    "music",
]


collection.add(
    documents=documents,
    metadatas=[{"genre": genre} for genre in genres],
    ids=[f"doc_{i}" for i in range(len(documents))],
)



# collection.add(
#     documents = list of documents/texts -> chunks
#     ids = list of ids for each document -> [f"doc_{i}" for i in range(len(chunks))]
#     metadatas = list of metadata for each document -> [{"page_number": chunk["page_number"], "source": chunk["source"]} for chunk in chunks]
# )

In [39]:
docs = ["chunk 1", "chunk 2"]
ids = ["id-1", "id-2"]

[f"id--{i}" for i in range(len(docs))]




['id--0', 'id--1']

In [47]:
query_results = collection.query(
    query_texts=["Find me some deleicious food"],
    n_results=3
    )


query_results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [54]:
query_results["documents"][0]

['Traditional Italian pizza is famous for its thin crust, fresh ingredients, and wood-fired ovens.',
 'Exploring the beautiful beaches and vibrant culture of Bali is a dream for many travelers.',
 'Regular exercise and a balanced diet are essential for maintaining good physical health.']

In [49]:
query_results["ids"]

[['doc_3', 'doc_1', 'doc_5']]

In [50]:
query_results["distances"]

[[0.7641699314117432, 0.8529417514801025, 0.8750079870223999]]